# Engine

Everything the notebooks share, each formula written once: the model, the grids, the band geometry, the currents and the band-edge tools. Nothing heavy runs here. Each notebook starts with `%run engine.ipynb` and then makes its own figures, without needing any other notebook.

| notebook | what it makes |
|---|---|
| `nn_tnn.ipynb` | Figs. 1-4: bands, Berry curvature, quantum metric, Chern numbers |
| `response.ipynb` | Fig. 5: injection and shift current of the four materials |
| `kp.ipynb` | Table II ($\kappa$ from the Löwdin reduction), Fig. 6 and Table V |
| `field.ipynb` | Figs. 7 and 8, and which field term does what |
| `edge.ipynb` | Figs. 9, 10 and 11, including the first-principles k.p points |
| `soc.ipynb` | dependence on the Ising coupling (not in the current draft) |

Long runs save their numbers in `data/`, and the paper's figures go to `figures/` as PDFs.

## Packages

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import gridspec
from matplotlib.lines import Line2D
from matplotlib.ticker import MaxNLocator

os.makedirs("data", exist_ok=True)
os.makedirs("figures", exist_ok=True)

## Paper style

Figure widths of the journal (text width `TW`, column width `CW`, in inches), fonts, colours and the two small helpers the figures share.

In [ ]:
TW, CW = 7.0, 3.4
plt.rcParams.update({
    "font.family": "serif", "font.serif": ["cmr10", "DejaVu Serif"], "mathtext.fontset": "cm",
    "axes.formatter.use_mathtext": True, "axes.unicode_minus": False,
    "font.size": 8, "axes.labelsize": 8, "axes.titlesize": 8, "legend.fontsize": 7,
    "xtick.labelsize": 7, "ytick.labelsize": 7,
    "axes.linewidth": 0.6, "lines.linewidth": 1.1, "lines.markersize": 3.5,
    "xtick.direction": "in", "ytick.direction": "in", "xtick.top": True, "ytick.right": True,
    "xtick.major.size": 3, "ytick.major.size": 3, "xtick.minor.size": 1.6, "ytick.minor.size": 1.6,
    "xtick.major.width": 0.6, "ytick.major.width": 0.6, "xtick.minor.width": 0.5, "ytick.minor.width": 0.5,
    "xtick.minor.visible": True, "ytick.minor.visible": True,
    "legend.frameon": False, "legend.handlelength": 1.8, "legend.borderaxespad": 0.4,
    "savefig.dpi": 600, "savefig.bbox": "tight", "savefig.pad_inches": 0.02,
    "pdf.fonttype": 42, "image.interpolation": "nearest",
})
mat_label = {"MoS2": r"MoS$_2$", "MoSe2": r"MoSe$_2$", "WS2": r"WS$_2$", "WSe2": r"WSe$_2$"}
mat_color = {"MoS2": "#0072B2", "MoSe2": "#D55E00", "WS2": "#009E73", "WSe2": "#CC79A7"}
BLUE, RED, GREY, ORANGE = "#0072B2", "#C0392B", "0.55", "#E69F00"
UNIT = r"\mathrm{A\,m\,V^{-2}\,s^{-1}}"
SUNIT = r"\mathrm{nm\,\mu A\,V^{-2}}"


def label(ax, s, x=0.0, y=1.02, **kw):
    kw.setdefault("va", "bottom")
    ax.text(x, y, r"$\mathbf{%s}$" % s, transform=ax.transAxes, fontsize=9, ha="left", **kw)


def zero(ax, v=False):
    ax.axhline(0, color="0.8", lw=0.5, zorder=0)
    if v:
        ax.axvline(0, color="0.8", lw=0.5, zorder=0)

## Constants

In [ ]:
mu_B = 5.7883818012e-5     # eV/T
g_s = 2
e = 1.602176634e-19        # C
hbar = 1.054571817e-34     # J s
m_e = 9.1093837015e-31     # kg

B80 = 0.08 / mu_B          # the field where mu_B B = 80 meV, used in Figs. 1-5 so the splitting is visible

materials = ["MoS2", "MoSe2", "WS2", "WSe2"]

## Tight-binding parameters

Liu *et al.*, PRB **88**, 085433 (2013), Tables II-IV, GGA and LDA fits. Lattice constant $a$ in Å, on-site energies and hoppings in eV. The TNN model uses the $a$ of the NN table, and the Ising coupling $\lambda$ is the same for both fits.

In [ ]:
NN_keys = ["a", "epsilon1", "epsilon2", "t0", "t1", "t2", "t11", "t12", "t22"]
NN_table = {
    "GGA": {"MoS2":  [3.190, 1.046, 2.104, -0.184, 0.401, 0.507, 0.218, 0.338, 0.057],
            "MoSe2": [3.326, 0.919, 2.065, -0.188, 0.317, 0.456, 0.211, 0.290, 0.130],
            "WS2":   [3.191, 1.130, 2.275, -0.206, 0.567, 0.536, 0.286, 0.384, -0.061],
            "WSe2":  [3.325, 0.943, 2.179, -0.207, 0.457, 0.486, 0.263, 0.329, 0.034]},
    "LDA": {"MoS2":  [3.129, 1.238, 2.366, -0.218, 0.444, 0.533, 0.250, 0.360, 0.047],
            "MoSe2": [3.254, 1.001, 2.239, -0.222, 0.350, 0.488, 0.244, 0.314, 0.129],
            "WS2":   [3.132, 1.355, 2.569, -0.238, 0.626, 0.557, 0.324, 0.405, -0.076],
            "WSe2":  [3.253, 1.124, 2.447, -0.242, 0.506, 0.514, 0.305, 0.353, 0.025]},
}

TNN_keys = ["epsilon1", "epsilon2", "t0", "t1", "t2", "t11", "t12", "t22",
            "r0", "r1", "r2", "r11", "r12", "u0", "u1", "u2", "u11", "u12", "u22"]
TNN_table = {
    "GGA": {"MoS2":  [0.683, 1.707, -0.146, -0.114, 0.506, 0.085, 0.162, 0.073, 0.060, -0.236, 0.067, 0.016, 0.087, -0.038, 0.046, 0.001, 0.266, -0.176, -0.150],
            "MoSe2": [0.684, 1.546, -0.146, -0.130, 0.432, 0.144, 0.117, 0.075, 0.039, -0.209, 0.069, 0.052, 0.060, -0.042, 0.036, 0.008, 0.272, -0.172, -0.150],
            "WS2":   [0.717, 1.916, -0.152, -0.097, 0.590, 0.047, 0.178, 0.016, 0.069, -0.261, 0.107, -0.003, 0.109, -0.054, 0.045, 0.002, 0.325, -0.206, -0.163],
            "WSe2":  [0.728, 1.655, -0.146, -0.124, 0.507, 0.117, 0.127, 0.015, 0.036, -0.234, 0.107, 0.044, 0.075, -0.061, 0.032, 0.007, 0.329, -0.202, -0.164]},
    "LDA": {"MoS2":  [0.820, 1.931, -0.176, -0.101, 0.531, 0.084, 0.169, 0.070, 0.070, -0.252, 0.084, 0.019, 0.093, -0.043, 0.047, 0.005, 0.304, -0.192, -0.162],
            "MoSe2": [0.715, 1.687, -0.154, -0.134, 0.437, 0.124, 0.119, 0.072, 0.048, -0.248, 0.090, 0.066, 0.045, -0.067, 0.041, 0.005, 0.327, -0.194, -0.151],
            "WS2":   [0.905, 2.167, -0.175, -0.090, 0.611, 0.043, 0.181, 0.008, 0.075, -0.282, 0.127, 0.001, 0.114, -0.063, 0.047, 0.004, 0.374, -0.224, -0.177],
            "WSe2":  [0.860, 1.892, -0.152, -0.125, 0.508, 0.094, 0.129, 0.009, 0.044, -0.278, 0.129, 0.059, 0.058, -0.090, 0.039, 0.001, 0.392, -0.224, -0.165]},
}

lambda_SOC = {"MoS2": 0.073, "MoSe2": 0.091, "WS2": 0.211, "WSe2": 0.228}


def params(mat, func="GGA", model="TNN"):
    if model == "NN":
        return dict(zip(NN_keys, NN_table[func][mat]))
    p = dict(zip(TNN_keys, TNN_table[func][mat]))
    p["a"] = NN_table[func][mat][0]
    return p

## NN Hamiltonian

Basis $(d_{z^2}, d_{xy}, d_{x^2-y^2})$, $\alpha = k_xa/2$, $\beta = \sqrt3k_ya/2$. We set $a = 1$, so $k$ is in units of $1/a$ and velocities in eV $a$.

In [ ]:
def H_NN(alpha, beta, p):
    epsilon1, epsilon2, t0, t1, t2, t11, t12, t22 = [p[k] for k in NN_keys[1:]]

    h0 = 2 * t0 * (np.cos(2 * alpha) + 2 * np.cos(alpha) * np.cos(beta)) + epsilon1
    h1 = -2 * np.sqrt(3) * t2 * np.sin(alpha) * np.sin(beta) + 2j * t1 * (np.sin(2 * alpha) + np.sin(alpha) * np.cos(beta))
    h2 = 2 * t2 * (np.cos(2 * alpha) - np.cos(alpha) * np.cos(beta)) + 2 * np.sqrt(3) * 1j * t1 * np.cos(alpha) * np.sin(beta)
    h11 = 2 * t11 * np.cos(2 * alpha) + (t11 + 3 * t22) * np.cos(alpha) * np.cos(beta) + epsilon2
    h22 = 2 * t22 * np.cos(2 * alpha) + (3 * t11 + t22) * np.cos(alpha) * np.cos(beta) + epsilon2
    h12 = np.sqrt(3) * (t22 - t11) * np.sin(alpha) * np.sin(beta) + 4j * t12 * np.sin(alpha) * (np.cos(alpha) - np.cos(beta))

    H = np.array([[h0, h1, h2], [np.conj(h1), h11, h12], [np.conj(h2), np.conj(h12), h22]])
    return np.moveaxis(H, (0, 1), (-2, -1))

## TNN Hamiltonian

In [ ]:
def H_TNN(alpha, beta, p):
    epsilon1, epsilon2, t0, t1, t2, t11, t12, t22 = [p[k] for k in TNN_keys[:8]]
    r0, r1, r2, r11, r12, u0, u1, u2, u11, u12, u22 = [p[k] for k in TNN_keys[8:]]

    V0 = epsilon1 + 2 * t0 * (2 * np.cos(alpha) * np.cos(beta) + np.cos(2 * alpha)) + 2 * r0 * (2 * np.cos(3 * alpha) * np.cos(beta) + np.cos(2 * beta)) + 2 * u0 * (2 * np.cos(2 * alpha) * np.cos(2 * beta) + np.cos(4 * alpha))

    ReV1 = -2 * np.sqrt(3) * t2 * np.sin(alpha) * np.sin(beta) + 2 * (r1 + r2) * np.sin(3 * alpha) * np.sin(beta) - 2 * np.sqrt(3) * u2 * np.sin(2 * alpha) * np.sin(2 * beta)
    ImV1 = 2 * t1 * np.sin(alpha) * (2 * np.cos(alpha) + np.cos(beta)) + 2 * (r1 - r2) * np.sin(3 * alpha) * np.cos(beta) + 2 * u1 * np.sin(2 * alpha) * (2 * np.cos(2 * alpha) + np.cos(2 * beta))
    V1 = ReV1 + 1j * ImV1

    ReV2 = 2 * t2 * (np.cos(2 * alpha) - np.cos(alpha) * np.cos(beta)) - (2 / np.sqrt(3)) * (r1 + r2) * (np.cos(3 * alpha) * np.cos(beta) - np.cos(2 * beta)) + 2 * u2 * (np.cos(4 * alpha) - np.cos(2 * alpha) * np.cos(2 * beta))
    ImV2 = 2 * np.sqrt(3) * t1 * np.cos(alpha) * np.sin(beta) + (2 / np.sqrt(3)) * np.sin(beta) * (r1 - r2) * (np.cos(3 * alpha) + 2 * np.cos(beta)) + 2 * np.sqrt(3) * u1 * np.cos(2 * alpha) * np.sin(2 * beta)
    V2 = ReV2 + 1j * ImV2

    V11 = epsilon2 + (t11 + 3 * t22) * np.cos(alpha) * np.cos(beta) + 2 * t11 * np.cos(2 * alpha) + 4 * r11 * np.cos(3 * alpha) * np.cos(beta) + 2 * (r11 + np.sqrt(3) * r12) * np.cos(2 * beta) + (u11 + 3 * u22) * np.cos(2 * alpha) * np.cos(2 * beta) + 2 * u11 * np.cos(4 * alpha)

    ReV12 = np.sqrt(3) * (t22 - t11) * np.sin(alpha) * np.sin(beta) + 4 * r12 * np.sin(3 * alpha) * np.sin(beta) + np.sqrt(3) * (u22 - u11) * np.sin(2 * alpha) * np.sin(2 * beta)
    ImV12 = 4 * t12 * np.sin(alpha) * (np.cos(alpha) - np.cos(beta)) + 4 * u12 * np.sin(2 * alpha) * (np.cos(2 * alpha) - np.cos(2 * beta))
    V12 = ReV12 + 1j * ImV12

    V22 = epsilon2 + (3 * t11 + t22) * np.cos(alpha) * np.cos(beta) + 2 * t22 * np.cos(2 * alpha) + 2 * r11 * (2 * np.cos(3 * alpha) * np.cos(beta) + np.cos(2 * beta)) + (2 / np.sqrt(3)) * r12 * (4 * np.cos(3 * alpha) * np.cos(beta) - np.cos(2 * beta)) + (3 * u11 + u22) * np.cos(2 * alpha) * np.cos(2 * beta) + 2 * u22 * np.cos(4 * alpha)

    H = np.array([[V0, V1, V2], [np.conj(V1), V11, V12], [np.conj(V2), np.conj(V12), V22]])
    return np.moveaxis(H, (0, 1), (-2, -1))


def hamiltonian(model, p, k_x, k_y):
    alpha = 0.5 * k_x
    beta = (np.sqrt(3) / 2) * k_y
    if model == "NN":
        return H_NN(alpha, beta, p)
    return H_TNN(alpha, beta, p)

## Spin-orbit coupling and the Zeeman field

Spin is conserved, so the $6\times6$ Hamiltonian is two $3\times3$ blocks, and we diagonalize each block on its own. The field couples to the on-site orbital moment ($L_z$ mixes $d_{xy}$ and $d_{x^2-y^2}$) and to the spin:
$$H_{\uparrow} = H + \frac{\lambda}{2}L_z + \mu_B B\,L_z + \frac{g_s}{2}\mu_B B, \qquad H_{\downarrow} = H - \frac{\lambda}{2}L_z + \mu_B B\,L_z - \frac{g_s}{2}\mu_B B.$$
In each block band 0 is the valence band and bands 1 and 2 are conduction bands ($T = 0$, chemical potential in the gap).

In [ ]:
L_z = np.array([[0, 0, 0], [0, 0, 2j], [0, -2j, 0]])


def zeeman(H, lam, B):
    up = H + (lam / 2) * L_z + mu_B * B * L_z + (g_s / 2) * mu_B * B * np.eye(3)
    dn = H - (lam / 2) * L_z + mu_B * B * L_z - (g_s / 2) * mu_B * B * np.eye(3)
    return up, dn

## k-grids

Every k-sum runs over a list of k-points. `slanted_grid` and `disk` return `(k_x, k_y, dA)`, the points and the area each one stands for.

* The whole zone: `slanted_grid(N)` is an $N\times N$ mesh along the two reciprocal lattice vectors $\mathbf{b}_1 = 2\pi(1, -1/\sqrt3)$ and $\mathbf{b}_2 = 2\pi(0, 2/\sqrt3)$. The mesh is slanted because $\mathbf{b}_1$ and $\mathbf{b}_2$ are 120° apart. It fills a parallelogram with the area of the hexagonal zone, and since the bands repeat from one zone to the next it covers every state exactly once, so no mask is needed.
* Near the band edge only the valleys matter: `disk(R, h)` is a square grid of spacing $h$ inside $|\mathbf{q}| \le R$, used around $K = (4\pi/3, 0)$ and $K' = -K$.

The disk contains $-\mathbf{q}$ for every $\mathbf{q}$, so time reversal pairs the two valleys point by point. At $B = 0$ each valley alone gives 3 to 300 times the field-induced signal and the two valleys have to cancel, so the pairing must be exact.


In [ ]:
K_len = 4 * np.pi / 3


def slanted_grid(N):
    f = (np.arange(N) + 0.5) / N
    f_1, f_2 = np.meshgrid(f, f, indexing="ij")
    k_x = 2 * np.pi * f_1
    k_y = 2 * np.pi * (2 * f_2 - f_1) / np.sqrt(3)
    return k_x.ravel(), k_y.ravel(), 8 * np.pi**2 / np.sqrt(3) / N**2


def disk(R, h):
    n = int(np.floor(R / h + 1e-9))
    i, j = np.meshgrid(np.arange(-n, n + 1), np.arange(-n, n + 1))
    inside = i**2 + j**2 <= (R / h)**2 + 1e-9
    return h * i[inside], h * j[inside], h**2


## Two-band k.p model

The warped Dirac model for valley $\tau$ and spin $s$,
$$H = \mathbf{d}\cdot\boldsymbol\sigma, \quad d_x = v\tau k_x + \kappa(k_x^2 - k_y^2), \quad d_y = vk_y - 2\kappa\tau k_xk_y, \quad d_z = m_D + wk^2 - \tau(b + s\lambda/2),$$
with $b = \mu_BB$. The field only enters through $b$ in $d_z$. `kp.ipynb` takes the parameters from the Löwdin reduction, `edge.ipynb` from first principles.

In [ ]:
def kp_hamiltonian(q_xv, q_yv, tau, s, b, kp):
    d_x = kp["v"] * tau * q_xv + kp["kappa"] * (q_xv**2 - q_yv**2)
    d_y = kp["v"] * q_yv - 2 * kp["kappa"] * tau * q_xv * q_yv
    d_z = kp["mD"] + kp["w"] * (q_xv**2 + q_yv**2) - tau * (b + s * kp["lam"] / 2)
    H = np.array([[d_z, d_x - 1j * d_y], [d_x + 1j * d_y, -d_z]])
    return np.moveaxis(H, (0, 1), (-2, -1))

## Band geometry

Every $k$-derivative is a central difference with a small step $dk = 10^{-4}/a$ (`central`),
$$\partial_{k_x}f(\mathbf{k}) \approx \frac{f(k_x + dk, k_y) - f(k_x - dk, k_y)}{2\,dk},$$
which recomputes $f$ next to each point, so it works the same on any grid. Below, `H` is a function of $(k_x, k_y)$.

In one block `eigh` gives $E_n$ and $|u_n\rangle$, and $v^a_{nm} = \langle u_n|\partial_aH|u_m\rangle$ (`bands`). From these:

* Berry curvature and orbital moment (`berry_and_moment`), $\Omega_n$ in units of $a^2$ and $M_n$ in Bohr magnetons, with $\kappa_{\rm orb} = m_ea^2/\hbar^2$ per eV ($1.336$ eV$^{-1}$ for MoS$_2$):
$$\Omega_n = -2\,\mathrm{Im}\sum_{m\ne n}\frac{v^x_{nm}v^y_{mn}}{(E_n - E_m)^2}, \qquad M_n = -\kappa_{\rm orb}\,2\,\mathrm{Im}\sum_{m\ne n}\frac{v^x_{nm}v^y_{mn}}{E_n - E_m};$$
* the interband quantum metric between the valence band $v$ (band 0) and a conduction band $c$ (`metric`), with the bare energies:
$$g^{bc}_{cv} = \frac{\mathrm{Re}[v^b_{vc}v^c_{cv}]}{(E_c - E_v)^2};$$
* the shift-current connection in projector form (`shift_connection`), with $P_n = |u_n\rangle\langle u_n|$,
$$C^{mn}_{a;bc} = \mathrm{Tr}\big[P_n\,\partial_bP_m\,\partial_a\partial_cP_n\big] + \mathrm{Tr}\big[P_n\,\partial_bP_m\,\partial_aP_m\,\partial_cP_n\big],$$
made antisymmetric in $m \leftrightarrow n$ ($\bar C^{mn}$); it returns $\mathrm{Im}\,\bar C^{cv}$ for each conduction band. The symmetrization in $b \leftrightarrow c$ happens in `C3_average`.

In [ ]:
dk = 1e-4      # step of the central differences (1/a)


def central(f, k_x, k_y):
    df_dkx = (f(k_x + dk, k_y) - f(k_x - dk, k_y)) / (2 * dk)
    df_dky = (f(k_x, k_y + dk) - f(k_x, k_y - dk)) / (2 * dk)
    return df_dkx, df_dky


def bands(H, k_x, k_y):
    E, V = np.linalg.eigh(H(k_x, k_y))
    dH_dkx, dH_dky = central(H, k_x, k_y)
    v_x = np.einsum('...kn, ...kl, ...lm -> ...nm', V.conj(), dH_dkx, V)
    v_y = np.einsum('...kn, ...kl, ...lm -> ...nm', V.conj(), dH_dky, V)
    return E, V, v_x, v_y


def berry_and_moment(E, v_x, v_y, a):
    dE = E[..., :, None] - E[..., None, :]
    n = np.arange(E.shape[-1])
    dE[..., n, n] = np.inf                                    # leave out m = n
    vv = np.einsum('...nm, ...mn -> ...nm', v_x, v_y)         # v^x_nm v^y_mn
    Omega = -2 * (vv / dE**2).sum(axis=-1).imag
    kappa_orb = m_e * (a * 1e-10)**2 / hbar**2 * e
    M = -kappa_orb * 2 * (vv / dE).sum(axis=-1).imag
    return Omega, M


def metric(E, v_x, v_y, c):
    v_vc = np.stack([v_x[..., 0, c], v_y[..., 0, c]], axis=-1)
    v_cv = np.stack([v_x[..., c, 0], v_y[..., c, 0]], axis=-1)
    return np.einsum('...b, ...c -> ...bc', v_vc, v_cv).real / (E[..., c] - E[..., 0])[..., None, None]**2


def shift_connection(H, k_x, k_y):
    def projectors(x, y):
        E, V = np.linalg.eigh(H(x, y))
        return np.einsum('...kn, ...ln -> ...nkl', V, V.conj())

    P = projectors(k_x, k_y)
    dP = np.stack(central(projectors, k_x, k_y))
    ddP = np.stack([np.stack(central(lambda x, y: central(projectors, x, y)[i], k_x, k_y)) for i in range(2)])
    C = (np.einsum('...npq, b...mqr, ac...nrp -> ...mnabc', P, dP, ddP, optimize=True)
         + np.einsum('...npq, b...mqr, a...mrs, c...nsp -> ...mnabc', P, dP, dP, dP, optimize=True))
    C = (C - np.einsum('...mnabc -> ...nmabc', C)) / 2          # antisymmetric in m, n
    return C[:, 1:, 0].imag                                     # m = c (conduction bands), n = v (band 0)


## The currents

The magnetic injection current, Eq. (6) of the paper,
$$\eta^{a;bc}(\omega) = -\frac{2\pi e^3a}{\hbar^2}\int\frac{d^2k}{(2\pi)^2}\sum_{c} D_v\,\Delta^a_{cv}\,g^{bc}_{cv}\,\delta_\nu(\varepsilon_{cv} - \hbar\omega), \qquad \delta_\nu(x) = \frac{\nu/\pi}{x^2 + \nu^2},$$
and the shift current,
$$\sigma^{a;(bc)}_{\rm shift} = -2\pi\int\frac{d^2k}{(2\pi)^2}\sum_c D_v\,\mathrm{Im}\,\bar C^{cv}_{a;(bc)}\,\delta_\nu(\hbar\omega - \varepsilon_{cv})$$
(code units, $e = \hbar = a = 1$). Besides entering the Hamiltonian, the field acts on the orbital moment $M_n$ in three places:

1. energy shift $\varepsilon_n = E_n - \mu_BB\,M_n$, only in the resonance $\delta_\nu$;
2. velocity shift $\Delta^a_{cv} = (\partial_aE_c - \mu_BB\,\partial_aM_c) - (\partial_aE_v - \mu_BB\,\partial_aM_v)$, with $\partial_aE_n = v^a_{nn}$;
3. phase-space factor $D_v = 1 + (ea^2/\hbar)\,B\,\Omega_v$ of the valence band ($ea^2/\hbar = 1.55\times10^{-4}$ T$^{-1}$ for MoS$_2$).

`field_terms` gives the three, each with its own field (0 switches it off). `currents` takes the Hamiltonian blocks as functions of $(k_x, k_y)$ (the two spin blocks of the tight-binding model, or the k.p blocks), sums over the k-points, the blocks and the conduction bands, and returns $\eta$ and $\sigma_{\rm shift}$, symmetric in $b \leftrightarrow c$ (linear polarization) and averaged over the three $C_3$ rotations (`C3_average`). Each switch turns one field term on or off; all three are on for every tight-binding figure from Fig. 7 on.

`eta_mag` does this for a tight-binding material on a grid centred at each point of `centers` (`[0]` for the whole zone, `[K, -K]` for the two valleys). It goes through the k-points in chunks, which keeps the memory small on the fine disk grids. `mirror` fills in negative fields from $\eta(-B) = -\eta(B)$. `eta_SI(a)` $= -2e^3a/\hbar^2$ turns $\eta$ into A m V$^{-2}$ s$^{-1}$, and `shift_SI(a)` $= e^2a/\hbar$ turns $\sigma_{\rm shift}$ into nm µA V$^{-2}$.

In [ ]:
def field_terms(H, k_x, k_y, E, v_x, v_y, Omega, M, a, B_E, B_V, B_D):
    def moment(x, y):
        E, V, v_x, v_y = bands(H, x, y)
        return berry_and_moment(E, v_x, v_y, a)[1]

    dM_dkx, dM_dky = central(moment, k_x, k_y) if B_V != 0 else (0, 0)
    E_res = E - mu_B * B_E * M                                                   # 1. energy shift
    dE_dkx = np.einsum('...nn -> ...n', v_x).real - mu_B * B_V * dM_dkx          # 2. velocity shift
    dE_dky = np.einsum('...nn -> ...n', v_y).real - mu_B * B_V * dM_dky
    D_v = 1 + e * (a * 1e-10)**2 / hbar * B_D * Omega[..., 0]                    # 3. phase-space factor
    return E_res, dE_dkx, dE_dky, D_v


def currents(blocks, k_x, k_y, dA, omega, nu, a=1.0, B=0.0,
             energy_shift=False, velocity_shift=False, phase_space=False, shift=False):
    B_E = B if energy_shift else 0
    B_V = B if velocity_shift else 0
    B_D = B if phase_space else 0
    eta = np.zeros((len(omega), 2, 2, 2))
    sigma = np.zeros((len(omega), 2, 2, 2))

    for H in blocks:
        E, V, v_x, v_y = bands(H, k_x, k_y)
        Omega, M = berry_and_moment(E, v_x, v_y, a)
        E_res, dE_dkx, dE_dky, D_v = field_terms(H, k_x, k_y, E, v_x, v_y, Omega, M, a, B_E, B_V, B_D)
        if shift:
            C = shift_connection(H, k_x, k_y)

        for c in range(1, E.shape[-1]):                     # each conduction band
            eps = E_res[:, c] - E_res[:, 0]
            Delta = np.stack([dE_dkx[:, c] - dE_dkx[:, 0], dE_dky[:, c] - dE_dky[:, 0]], axis=-1)
            X_eta = np.einsum('p, pa, pbc -> pabc', D_v, Delta, metric(E, v_x, v_y, c))
            if shift:
                X_shift = np.einsum('p, pabc -> pabc', D_v, C[:, c - 1])
            for w in range(len(omega)):
                delta = (nu / np.pi) / ((omega[w] - eps)**2 + nu**2)
                eta[w] += np.einsum('p, pabc -> abc', delta, X_eta, optimize=True)
                if shift:
                    sigma[w] += np.einsum('p, pabc -> abc', delta, X_shift, optimize=True)

    eta = C3_average(np.pi * dA / (2 * np.pi)**2 * eta)
    sigma = C3_average(-2 * np.pi * dA / (2 * np.pi)**2 * sigma)
    return eta, sigma


def C3_average(T):
    T = (T + T.transpose(0, 1, 3, 2)) / 2                 # linear polarization: symmetric in b, c
    T_avg = 0
    for n in range(3):                                     # average over the three C3 rotations
        R = np.array([[np.cos(2 * np.pi * n / 3), -np.sin(2 * np.pi * n / 3)],
                      [np.sin(2 * np.pi * n / 3), np.cos(2 * np.pi * n / 3)]])
        T_avg = T_avg + np.einsum('ia, jb, kc, wabc -> wijk', R, R, R, T) / 3
    return T_avg


def eta_mag(mat, model, B, omega, nu, grid, centers=[0.0], func="GGA", lam=None,
            energy_shift=True, velocity_shift=True, phase_space=True, shift=False, chunk=20000):
    p = params(mat, func, model)
    if lam is None:
        lam = lambda_SOC[mat]
    k_x, k_y, dA = grid
    blocks = [lambda x, y, s=s: zeeman(hamiltonian(model, p, x, y), lam, B)[s] for s in (0, 1)]      # spin up, spin down
    eta, sigma = 0, 0
    for k_x0 in centers:
        for start in range(0, len(k_x), chunk):
            eta_s, sigma_s = currents(blocks, k_x0 + k_x[start:start + chunk], k_y[start:start + chunk], dA, omega, nu, p["a"], B,
                                      energy_shift, velocity_shift, phase_space, shift)
            eta, sigma = eta + eta_s, sigma + sigma_s
    if shift:
        return eta, sigma
    return eta


def mirror(B, eta):
    # time reversal gives eta(-B) = -eta(B), so runs at B >= 0 cover the whole field range
    return np.concatenate([-B[:0:-1], B]), np.concatenate([-eta[:0:-1], eta])


def eta_SI(a):
    return -2 * e**3 * (a * 1e-10) / hbar**2      # code units -> A m V^-2 s^-1


def shift_SI(a):
    return e**2 * (a * 1e-10) / hbar / 1e-15       # code units -> nm uA V^-2


## k.p currents

The same `currents` for the k.p model, summed over the spin-valley locked pairs $(\tau, s) = (+,+)$ and $(-,-)$, on a disk of radius $R$ with $N_k$ points across. The field is only in the Hamiltonian, so the moment terms stay off.

In [ ]:
def kp_currents(kp, b, omega, nu, R, N_k, pairs=[(1, 1), (-1, -1)], shift=False):
    k_x, k_y, dA = disk(R, 2 * R / (N_k - 1))
    blocks = [lambda x, y, tau=tau, s=s: kp_hamiltonian(x, y, tau, s, b, kp) for tau, s in pairs]
    eta, sigma = currents(blocks, k_x, k_y, dA, omega, nu, shift=shift)
    if shift:
        return eta, sigma
    return eta


## Band gap, band width and the Löwdin reduction

$E_g$ is the smallest direct gap on a fine line through $K$ at $B = 0$ (both blocks), and $E_{\max}$ the largest interband energy $E_2 - E_0$ over the zone.

The Löwdin reduction at $K$ ($\tau = +1$, spin-up block, which holds the band edge) uses $|c\rangle = |d_{z^2}\rangle$, $|v\rangle = (|d_{x^2-y^2}\rangle + i|d_{xy}\rangle)/\sqrt2$ and the remote $|u\rangle = (|d_{x^2-y^2}\rangle - i|d_{xy}\rangle)/\sqrt2$. $L_z$ is diagonal in this basis, so spin-orbit coupling only moves the energies at $K$: $E_v = E_v^0 + \lambda$, $E_u = E_u^0 - \lambda$. To second order,
$$H^{\rm eff}_{ij} = H_{ij} + \frac12H_{iu}H_{uj}\left[\frac{1}{E_i - E_u} + \frac{1}{E_j - E_u}\right].$$
Fitting $H^{\rm eff}_{cv}(\mathbf{q}) = t(q_x - iq_y) + \kappa(q_x + iq_y)^2 + \dots$ on small rings around $K$ gives $v = at$ and the trigonal warping $\kappa$ (times $a^2$ for eV Å$^2$). Fitting $H^{\rm eff}_{cc} = E_c + \gamma_cq^2 + \dots$ and $H^{\rm eff}_{vv} = E_v + \gamma_vq^2 + \dots$ gives $w = (\gamma_c - \gamma_v)/2$ and $m_D = (E_c - E_v)/2$. The extra fit terms keep the ones we use unbiased. `kp.ipynb` uses it for Table II and Fig. 6, `edge.ipynb` for Fig. 11.

In [ ]:
def gap(mat, model="TNN", func="GGA", lam=None):
    p = params(mat, func, model)
    if lam is None:
        lam = lambda_SOC[mat]
    q = np.linspace(-0.05, 0.05, 401)
    gaps = []
    for H in zeeman(hamiltonian(model, p, K_len + q, np.zeros_like(q)), lam, 0):
        E = np.linalg.eigvalsh(H)
        gaps.append((E[:, 1] - E[:, 0]).min())
    return min(gaps)


def E_max(mat, model="TNN", func="GGA"):
    p = params(mat, func, model)
    k = np.linspace(-2 * np.pi, 2 * np.pi, 401)      # covers the whole zone
    k_xv, k_yv = np.meshgrid(k, k)
    widths = []
    for H in zeeman(hamiltonian(model, p, k_xv, k_yv), lambda_SOC[mat], 0):
        E = np.linalg.eigvalsh(H)
        widths.append((E[..., 2] - E[..., 0]).max())
    return max(widths)


def lowdin(mat, func="GGA", model="TNN", lam=None):
    p = params(mat, func, model)
    if lam is None:
        lam = lambda_SOC[mat]
    c = np.array([1, 0, 0])
    v = np.array([0, 1j, 1]) / np.sqrt(2)
    u = np.array([0, -1j, 1]) / np.sqrt(2)
    U = np.stack([c, v, u], axis=1)

    r = np.repeat([0.002, 0.004, 0.006, 0.01], 64)                      # four rings around K
    phi = np.tile(np.linspace(0, 2 * np.pi, 64, endpoint=False), 4)
    q_x, q_y = r * np.cos(phi), r * np.sin(phi)

    H_K = U.conj().T @ zeeman(hamiltonian(model, p, np.array([K_len]), np.array([0.0])), lam, 0)[0][0] @ U
    E_c, E_v, E_u = H_K.diagonal().real
    H = U.conj().T @ zeeman(hamiltonian(model, p, K_len + q_x, q_y), lam, 0)[0] @ U

    H_cv = H[:, 0, 1] + 0.5 * H[:, 0, 2] * H[:, 2, 1] * (1 / (E_c - E_u) + 1 / (E_v - E_u))
    H_cc = H[:, 0, 0].real + np.abs(H[:, 0, 2])**2 / (E_c - E_u)
    H_vv = H[:, 1, 1].real + np.abs(H[:, 1, 2])**2 / (E_v - E_u)

    z = q_x - 1j * q_y
    fit_cv = np.linalg.lstsq(np.stack([z, z.conj()**2, r**2 * z, z.conj(), z**2], axis=1), H_cv, rcond=None)[0]
    X = np.stack([np.ones_like(r), r**2, q_x * (q_x**2 - 3 * q_y**2), r**4], axis=1)
    fit_c = np.linalg.lstsq(X, H_cc, rcond=None)[0]
    fit_v = np.linalg.lstsq(X, H_vv, rcond=None)[0]
    return dict(v=fit_cv[0].real, kappa=fit_cv[1].real, w=(fit_c[1] - fit_v[1]) / 2, mD=(E_c - E_v) / 2, lam=lam, a=p["a"])